# Pitch técnico — Agente imobiliário multiagente

Este notebook apresenta o funcionamento, as vantagens, as facilidades e o caminho de escalonamento do sistema. As integrações com QuintoAndar, ZAP, Coelho da Fonseca e WhatsApp são **simuladas** pelo dropdown de origem do chat.

## 1. Funcionamento

O lead entra por um canal. O supervisor do LangGraph encaminha a mensagem para busca, avaliação, jurídico, financiamento ou vendas. A busca usa RAG híbrido (BM25 + Chroma) e o catálogo em SQLite. Nome e e-mail criam um lead com uma única conversa. Visitas geram arquivo ICS. O relatório lê o mesmo funil e sugere ações nos bairros mais buscados.

```mermaid
flowchart LR
  canal[Site_App_WhatsApp_Portais] --> chat[Chat]
  chat --> supervisor[Supervisor]
  supervisor --> especialistas[Especialistas]
  especialistas --> rag[RAG_hibrido]
  especialistas --> crm[CRM_e_agenda]
  crm --> relatorio[Relatorio]
```

## 2. Vantagens e benefícios

- **Custo local:** Llama 3.1 8B via Ollama, sem cobrança por token.
- **Continuidade:** um e-mail reabre a mesma conversa.
- **Gestão no mesmo fluxo:** funil, agenda e sugestões de venda usam os dados que o chat acabou de gravar.
- **Catálogo misto:** 50 imóveis residenciais e 50 empresariais, com fotos do tipo do imóvel.

## 3. Facilidades

- Seeds reproduzíveis para imóveis, RAG, vendedores e leads.
- Interface Streamlit em um comando.
- E-mail de visita em modo mock (arquivo `.ics`), sem SMTP obrigatório.
- Troca de canal da demonstração por um dropdown, sem nova integração.

## 4. Escalonamento

O grafo não precisa mudar para crescer:

1. Cada canal (site, app, WhatsApp, portal) entra como uma fila na frente do mesmo supervisor.
2. Mais vendedores usam o filtro que o relatório e a agenda já têm.
3. Ollama pode sair da máquina local para um servidor, e o Chroma pode ser trocado por outro banco vetorial.

## 5. Integração simulada

| Canal | O que um adaptador faria | O que a demo faz |
| --- | --- | --- |
| QuintoAndar | Webhook do portal vira mensagem no grafo, origem `quinto_andar` | Dropdown no chat |
| ZAP | Lead do anúncio entra com origem `zap` | Dropdown no chat |
| Coelho da Fonseca | Mesmo grafo para o canal de alto padrão | Dropdown no chat |
| WhatsApp | Mensagem do aplicativo entra no chat; nome e e-mail são pedidos na conversa | Dropdown no chat |
| Site e App | Formulário ou app publica no mesmo endpoint | Dropdown no chat |

Não há chamada real a essas APIs.

## 6. Números e origens

A célula abaixo lê o mesmo CRM da Home.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.db.models import init_db
from src.db.repository import list_properties, origem_distribution

init_db()
resid = len(list_properties(segmento="residencial", limit=200))
comerc = len(list_properties(segmento="empresarial", limit=200))
origens = origem_distribution() or {
    "site": 2, "app": 2, "whatsapp": 2,
    "quinto_andar": 2, "zap": 2, "coelho_da_fonseca": 2,
}
print(f"Residenciais: {resid} | Empresariais: {comerc} | Agentes: 8 | Canais simulados: 6")
serie = pd.Series(origens, name="leads")
serie.plot(kind="bar", title="Origem dos leads", color="#7c3aed")
plt.ylabel("Leads")
plt.tight_layout()
plt.show()


## 7. Tela inicial

Print da Home com o mesmo pitch, os cards e o gráfico de origens.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
shot = ROOT / "assets" / "screenshots" / "home_pitch.png"
if shot.exists():
    display(Image(filename=str(shot)))
else:
    print("Print ainda não gerado:", shot)
